# CropGuard: ResNet-50 Training on PlantVillage Dataset

This notebook prepares and trains the CropGuard plant disease detection model using ResNet-50 transfer learning on the PlantVillage dataset.

**Recommended Runtime:** Google Colab with GPU (Tesla T4 or better)

**Estimated Training Time:** 20-30 hours on Colab T4

## 1. Setup & Mount Google Drive (Colab only)

In [ ]:
# Mount Google Drive for dataset and model storage
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/plant_ai')  # Adjust path to your project directory
print(f"Current directory: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
# Install training dependencies
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q scikit-learn matplotlib seaborn pandas jupyter

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 3. Download & Extract PlantVillage Dataset

In [ ]:
# Install Kaggle CLI and configure credentials
!pip install -q kaggle

# Upload kaggle.json to Colab (or place in Google Drive)
# Option 1: Use Files upload (run this then upload your kaggle.json)
# from google.colab import files
# files.upload()

# Option 2: If kaggle.json is in your Drive
!mkdir -p ~/.kaggle
!cp /content/drive/MyDrive/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle configured")

In [ ]:
# Download and extract PlantVillage dataset from Kaggle
# Note: First time may take 10-30 minutes depending on internet speed

os.makedirs('data', exist_ok=True)

# Search and find PlantVillage on Kaggle, then use the dataset slug
# Example: jbuchheim/plantvillage
!kaggle datasets download -d jbuchheim/plantvillage -p data/ --unzip

# Verify extraction
import shutil
if os.path.exists('data/segmented_plant_dataset'):
    # Rename to standard PlantVillage for consistency
    shutil.move('data/segmented_plant_dataset', 'data/PlantVillage')
elif os.path.exists('data/PlantVillage'):
    print("PlantVillage dataset already in correct location")

print("Dataset extraction complete")

## 4. Verify Dataset Structure

In [ ]:
# Verify dataset structure and compute statistics
import os
from collections import Counter

data_root = 'data/PlantVillage'

if not os.path.exists(data_root):
    raise FileNotFoundError(f"Dataset not found at {data_root}. Please download first.")

# Count classes and images
class_names = sorted([d for d in os.listdir(data_root) if os.path.isdir(os.path.join(data_root, d))])
class_counts = {}
total_images = 0

for class_name in class_names:
    class_path = os.path.join(data_root, class_name)
    files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    class_counts[class_name] = len(files)
    total_images += len(files)

print(f"Total Classes: {len(class_names)}")
print(f"Total Images: {total_images}")
print(f"Dataset Size on Disk: {shutil.disk_usage(data_root)[0] / (1024**3):.2f} GB")
print(f"\nTop 10 Classes by Image Count:")

for cls, cnt in sorted(class_counts.items(), key=lambda x: -x[1])[:10]:
    print(f"  {cls}: {cnt} images")

# Check for class imbalance
counts = list(class_counts.values())
print(f"\nClass Distribution:")
print(f"  Min: {min(counts)} images")
print(f"  Max: {max(counts)} images")
print(f"  Mean: {sum(counts)/len(counts):.1f} images")

## 5. Clone CropGuard Repository & Setup Project

In [ ]:
# If you're not in the project directory yet, clone the repo
# Uncomment and modify as needed:
# !git clone https://github.com/your-org/plant_ai.git
# os.chdir('plant_ai')

# For this example, assuming we're already in the project directory
print(f"Project root: {os.getcwd()}")
print("Files in project:")
!ls -la | grep -E '(train|config|models|utils)'

## 6. Load & Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

# Display sample images from different classes
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

random_classes = random.sample(class_names, 6)

for idx, class_name in enumerate(random_classes):
    class_path = os.path.join(data_root, class_name)
    image_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    if image_files:
        img_path = os.path.join(class_path, random.choice(image_files))
        img = Image.open(img_path)
        
        axes[idx].imshow(img)
        axes[idx].set_title(class_name, fontsize=10)
        axes[idx].axis('off')

plt.tight_layout()
plt.savefig('sample_images.png', dpi=100, bbox_inches='tight')
plt.show()

print("Sample images displayed")

## 7. Import and Configure Project Modules

In [ ]:
# Add project root to sys.path
import sys
sys.path.insert(0, os.getcwd())

# Import project modules
from config import Config
from utils.data_loaders import get_data_loaders
from models.train import build_model, train_one_epoch, evaluate, save_checkpoint, unfreeze_for_finetuning
from models.utils import EarlyStopping

print("Project modules imported successfully")
print(f"Config - IMAGE_SIZE: {Config.IMAGE_SIZE}")
print(f"Config - BATCH_SIZE: {Config.BATCH_SIZE}")
print(f"Config - LEARNING_RATE: {Config.LEARNING_RATE}")

## 8. Create Data Loaders with Train/Val/Test Split

In [ ]:
# Create data loaders with stratified split
print("Creating data loaders...")

train_loader, val_loader, test_loader, class_names = get_data_loaders(
    data_dir=data_root,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    batch_size=32,  # Adjust based on GPU memory
    num_workers=4   # Colab can handle multiple workers
)

print(f"\nData Loaders Created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")
print(f"  Number of classes: {len(class_names)}")
print(f"\nClasses:")
for i, cls in enumerate(class_names, 1):
    print(f"  {i:2d}. {cls}")

## 9. Verify Data Augmentation Pipeline

In [ ]:
# Visualize augmented samples
from utils.transforms import get_train_transforms, get_val_test_transforms

print("Training Transforms:")
train_transforms = get_train_transforms()
print(train_transforms)

print("\nValidation/Test Transforms:")
val_transforms = get_val_test_transforms()
print(val_transforms)

# Get a sample batch
sample_batch, sample_labels = next(iter(train_loader))
print(f"\nSample batch shape: {sample_batch.shape}")
print(f"Sample labels shape: {sample_labels.shape}")

## 10. Build ResNet-50 Model

In [ ]:
# Build ResNet-50 with custom head
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

num_classes = len(class_names)
model = build_model(num_classes=num_classes, freeze_backbone=True)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture: ResNet-50")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")
print(f"Frozen Parameters: {total_params - trainable_params:,}")

## 11. Training Configuration

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time

# Training hyperparameters
EPOCHS_PHASE1 = 15
EPOCHS_PHASE2 = 5
LEARNING_RATE_PHASE1 = 0.001
LEARNING_RATE_PHASE2 = 0.00001
BATCH_SIZE = 32
SAVE_DIR = 'saved_models'

os.makedirs(SAVE_DIR, exist_ok=True)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()

print("Training Configuration:")
print(f"  Epochs Phase 1: {EPOCHS_PHASE1}")
print(f"  Epochs Phase 2: {EPOCHS_PHASE2}")
print(f"  Learning Rate Phase 1: {LEARNING_RATE_PHASE1}")
print(f"  Learning Rate Phase 2: {LEARNING_RATE_PHASE2}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Loss Function: CrossEntropyLoss")
print(f"  Optimizer: Adam")
print(f"  Scheduler: ReduceLROnPlateau")

## 12. PHASE 1: Train Classification Head

In [ ]:
# Phase 1: Train head with frozen backbone
print("="*60)
print("PHASE 1: Training Classification Head")
print("="*60)

optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE_PHASE1)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1, verbose=True)
early_stopping = EarlyStopping(patience=5)

best_val_loss = float('inf')
history_phase1 = []

for epoch in range(1, EPOCHS_PHASE1 + 1):
    start_time = time.time()
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    epoch_time = time.time() - start_time
    scheduler.step(val_loss)
    
    history_phase1.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    })
    
    print(f"Epoch {epoch}/{EPOCHS_PHASE1} ({epoch_time:.1f}s) | "
          f"TL: {train_loss:.4f} TA: {train_acc*100:.2f}% | "
          f"VL: {val_loss:.4f} VA: {val_acc*100:.2f}%")
    
    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, epoch, val_loss, class_names, SAVE_DIR, "best_head_model.pth")
        print(f"  → New best model saved!")
    
    # Early stopping check
    if early_stopping(val_loss):
        print("Early stopping triggered")
        break

print(f"\nPhase 1 Complete. Best validation loss: {best_val_loss:.4f}")

## 13. PHASE 2: Fine-tune Entire Network

In [ ]:
# Phase 2: Unfreeze and fine-tune full model
print("="*60)
print("PHASE 2: Fine-tuning Entire Network")
print("="*60)

unfreeze_for_finetuning(model)

# Recount trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters after unfreezing: {trainable_params:,}")

optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE_PHASE2)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=2, factor=0.1, verbose=True)
early_stopping = EarlyStopping(patience=5)

history_phase2 = []

for epoch in range(1, EPOCHS_PHASE2 + 1):
    start_time = time.time()
    
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    
    epoch_time = time.time() - start_time
    scheduler.step(val_loss)
    
    history_phase2.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc
    })
    
    print(f"Finetune Epoch {epoch}/{EPOCHS_PHASE2} ({epoch_time:.1f}s) | "
          f"TL: {train_loss:.4f} TA: {train_acc*100:.2f}% | "
          f"VL: {val_loss:.4f} VA: {val_acc*100:.2f}%")
    
    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, epoch, val_loss, class_names, SAVE_DIR, "best_model.pth")
        print(f"  → New best model saved!")
    
    # Early stopping check
    if early_stopping(val_loss):
        print("Early stopping triggered")
        break

print(f"\nPhase 2 Complete. Best validation loss: {best_val_loss:.4f}")

## 14. Visualize Training History

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
all_history = history_phase1 + history_phase2
epochs = [h['epoch'] for h in all_history]
train_losses = [h['train_loss'] for h in all_history]
val_losses = [h['val_loss'] for h in all_history]

axes[0].plot(epochs, train_losses, 'b-o', label='Train Loss')
axes[0].plot(epochs, val_losses, 'r-o', label='Val Loss')
axes[0].axvline(x=len(history_phase1), color='gray', linestyle='--', label='Phase 1 → 2')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
train_accs = [h['train_acc'] * 100 for h in all_history]
val_accs = [h['val_acc'] * 100 for h in all_history]

axes[1].plot(epochs, train_accs, 'b-o', label='Train Acc')
axes[1].plot(epochs, val_accs, 'r-o', label='Val Acc')
axes[1].axvline(x=len(history_phase1), color='gray', linestyle='--', label='Phase 1 → 2')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'training_curves.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"Training curves saved to {SAVE_DIR}/training_curves.png")

## 15. Evaluate on Test Set

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import numpy as np

# Load best model
best_model_path = os.path.join(SAVE_DIR, 'best_model.pth')
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded best model from {best_model_path}")

# Evaluate on test set
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Compute metrics
accuracy = accuracy_score(all_labels, all_preds)
precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro')

print("\n" + "="*60)
print("TEST SET EVALUATION")
print("="*60)
print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1 Score:  {f1*100:.2f}%")
print("="*60)

## 16. Plot Confusion Matrix

In [ ]:
import seaborn as sns

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"Confusion matrix saved to {SAVE_DIR}/confusion_matrix.png")

## 17. Export to ONNX

In [ ]:
# Export model to ONNX
print("Exporting model to ONNX...")

dummy_input = torch.randn(1, 3, 224, 224).to(device)
onnx_path = os.path.join(SAVE_DIR, 'model.onnx')

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=12,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"✓ Model exported to {onnx_path}")

# Save class names for ONNX predictor
class_names_path = os.path.join(SAVE_DIR, 'class_names.txt')
with open(class_names_path, 'w') as f:
    for name in class_names:
        f.write(name + '\n')

print(f"✓ Class names saved to {class_names_path}")

## 18. Validate ONNX Model

In [ ]:
import onnxruntime as ort
import numpy as np

# Load ONNX session
ort_session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
input_name = ort_session.get_inputs()[0].name

print(f"ONNX Model Loaded Successfully")
print(f"  Input name: {input_name}")
print(f"  Input shape: {ort_session.get_inputs()[0].shape}")
print(f"  Output name: {ort_session.get_outputs()[0].name}")

# Validate outputs match PyTorch
print("\nValidating ONNX outputs against PyTorch...")

model.eval()
with torch.no_grad():
    dummy_input_torch = torch.randn(1, 3, 224, 224).to(device)
    py_output = model(dummy_input_torch).cpu().numpy()

dummy_input_np = dummy_input_torch.cpu().numpy()
ort_outputs = ort_session.run(None, {input_name: dummy_input_np})[0]

# Check if outputs are close
if np.allclose(py_output, ort_outputs, rtol=1e-03, atol=1e-05):
    print("✓ ONNX output matches PyTorch output!")
else:
    max_diff = np.max(np.abs(py_output - ort_outputs))
    print(f"⚠ Output difference: {max_diff:.6f} (may be acceptable)")

print(f"\n✓ ONNX export and validation complete!")

## 19. Benchmark ONNX vs PyTorch Inference

In [ ]:
import time

print("Benchmarking Inference Speed...\n")

# Prepare test input
test_input = torch.randn(1, 3, 224, 224).to(device)
test_input_np = test_input.cpu().numpy()

# Warmup
for _ in range(10):
    with torch.no_grad():
        _ = model(test_input)
    _ = ort_session.run(None, {input_name: test_input_np})

# PyTorch benchmark
pytorch_times = []
for _ in range(100):
    start = time.perf_counter()
    with torch.no_grad():
        _ = model(test_input)
    pytorch_times.append(time.perf_counter() - start)

# ONNX benchmark
onnx_times = []
for _ in range(100):
    start = time.perf_counter()
    _ = ort_session.run(None, {input_name: test_input_np})
    onnx_times.append(time.perf_counter() - start)

pytorch_mean = np.mean(pytorch_times) * 1000
onnx_mean = np.mean(onnx_times) * 1000
speedup = pytorch_mean / onnx_mean

print("="*50)
print("Inference Speed Comparison")
print("="*50)
print(f"PyTorch (GPU) Latency:  {pytorch_mean:.2f} ms")
print(f"ONNX (CPU) Latency:     {onnx_mean:.2f} ms")
print(f"Speedup (ONNX/PyTorch): {speedup:.2f}x")
print("="*50)

## 20. Save Training Summary

In [ ]:
import json
from datetime import datetime

# Create training summary
summary = {
    'timestamp': datetime.now().isoformat(),
    'dataset': {
        'num_classes': len(class_names),
        'total_images': total_images,
        'class_names': class_names
    },
    'training': {
        'epochs_phase1': len(history_phase1),
        'epochs_phase2': len(history_phase2),
        'learning_rate_phase1': LEARNING_RATE_PHASE1,
        'learning_rate_phase2': LEARNING_RATE_PHASE2,
        'batch_size': BATCH_SIZE
    },
    'results': {
        'test_accuracy': float(accuracy),
        'test_precision': float(precision),
        'test_recall': float(recall),
        'test_f1': float(f1)
    },
    'files': {
        'pytorch_model': 'best_model.pth',
        'onnx_model': 'model.onnx',
        'class_names': 'class_names.txt',
        'training_curves': 'training_curves.png',
        'confusion_matrix': 'confusion_matrix.png'
    }
}

summary_path = os.path.join(SAVE_DIR, 'training_summary.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"✓ Training summary saved to {summary_path}")
print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"\nGenerated Files:")
print(f"  - {SAVE_DIR}/best_model.pth")
print(f"  - {SAVE_DIR}/model.onnx")
print(f"  - {SAVE_DIR}/class_names.txt")
print(f"  - {SAVE_DIR}/training_curves.png")
print(f"  - {SAVE_DIR}/confusion_matrix.png")
print(f"  - {SAVE_DIR}/training_summary.json")
print("\nNext Steps:")
print("1. Download these files from Google Drive")
print("2. Copy to project's saved_models/ directory")
print("3. Update backend/app/api/routes.py to use ONNX model")
print("4. Test end-to-end prediction with frontend")
print("="*60)

## 21. Download Files from Colab (if needed)

In [ ]:
# If running on Colab and not using Google Drive directly,
# use this to download files

# from google.colab import files
# print("Downloading saved_models directory...")
# !cd saved_models && ls -lah
# files.download('saved_models/best_model.pth')
# files.download('saved_models/model.onnx')
# files.download('saved_models/training_curves.png')
# files.download('saved_models/confusion_matrix.png')

print("Files are saved in the project directory")
print("If using Google Drive (mounted), files are automatically saved there.")